# Which segments carry the most variance in fraud rate?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [1]:
import plotly.graph_objects as go
import polars as pl
from IPython.display import Markdown, display
from plotly.subplots import make_subplots

t = pl.scan_csv("../../kaggle/raw/train_transaction.csv")
i = pl.scan_csv("../../kaggle/raw/train_identity.csv")
df = t.join(i, on="TransactionID", how="left")
ctx = pl.SQLContext(df=df)


In [2]:
total_rows = df.select(pl.len()).collect().item()

ent_stats = df.select([
    pl.col("card1").n_unique().alias("card1_dist"),
    pl.col("card1").is_not_null().sum().alias("card1_cov"),
    pl.col("DeviceInfo").n_unique().alias("dev_dist"),
    pl.col("DeviceInfo").is_not_null().sum().alias("dev_cov"),
    pl.col("ProductCD").n_unique().alias("prod_dist"),
    pl.col("ProductCD").is_not_null().sum().alias("prod_cov")
]).collect().row(0)

ent_data = [
    {"Entity": "`card1`", "Distinct values": f"{ent_stats[0]:,}", "Coverage": f"{ent_stats[1]/total_rows*100:.1f}%"},
    {"Entity": "`DeviceInfo`", "Distinct values": f"{ent_stats[2]:,}", "Coverage": f"{ent_stats[3]/total_rows*100:.1f}% ({ent_stats[3]:,} rows)"},
    {"Entity": "`ProductCD`", "Distinct values": f"{ent_stats[4]:,}", "Coverage": f"{ent_stats[5]/total_rows*100:.1f}%"}
]
display(Markdown(pl.DataFrame(ent_data).to_pandas().to_markdown(index=False)))


| Entity       |   Distinct values | Coverage             |
|:-------------|------------------:|:---------------------|
| `card1`      |            13,553 | 100.0%               |
| `DeviceInfo` |             1,787 | 20.1% (118,666 rows) |
| `ProductCD`  |                 5 | 100.0%               |

## High-Variance Segments

**By product:**

Product C carries 5.7× the fraud rate of product W on a tenth of the volume.

In [3]:
df_prod = ctx.execute("""
SELECT 
    ProductCD,
    COUNT(*) as Transactions,
    SUM(isFraud) / COUNT(*) as Fraud_rate
FROM df
GROUP BY ProductCD
ORDER BY Transactions DESC
""").collect()

df_prod_md = df_prod.with_columns([
    pl.col("Fraud_rate").map_elements(lambda x: f"**{x*100:.2f}%**" if x > 0.1 else f"{x*100:.2f}%", return_dtype=pl.String)
])
display(Markdown(df_prod_md.to_pandas().to_markdown(index=False)))

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=df_prod["ProductCD"], y=df_prod["Transactions"], name="Volume", opacity=0.4, marker_color="#636efa"), secondary_y=False)
fig.add_trace(go.Scatter(x=df_prod["ProductCD"], y=df_prod["Fraud_rate"], name="Fraud Rate", mode="lines+markers", line={"color": "red", "width": 3}), secondary_y=True)
fig.update_layout(title="Transactions and Fraud Rate by ProductCD", hovermode="x unified", template="plotly_white", width=800)
fig.update_yaxes(title_text="Transactions", secondary_y=False)
fig.update_yaxes(title_text="Fraud Rate", tickformat=".1%", secondary_y=True)
fig.show()


| ProductCD   |   Transactions | Fraud_rate   |
|:------------|---------------:|:-------------|
| W           |         439670 | 2.04%        |
| C           |          68519 | **11.69%**   |
| R           |          37699 | 3.78%        |
| H           |          33024 | 4.77%        |
| S           |          11628 | 5.90%        |

**By device type:**


The presence of device information is itself worth 3–5× on the fraud rate. That is a
strong signal and a trap in equal measure: it is why the device velocity aggregate had to
be null-guarded rather than left to count across every device-less row at once (see the
decision log, 2026-08-10), and why `DeviceInfo` stays in the model as a feature so the
"missing" signal reaches the model through the column that honestly carries it.



In [4]:
df_dev = df.with_columns(
    pl.when(pl.col("DeviceInfo").is_null() | (pl.col("DeviceInfo") == ""))
    .then(pl.lit("_absent_"))
    .when(pl.col("DeviceType").is_null())
    .then(pl.lit("unknown"))
    .otherwise(pl.col("DeviceType"))
    .alias("Device_Category")
).group_by("Device_Category").agg([
    pl.len().alias("Transactions"),
    (pl.col("isFraud").sum() / pl.len()).alias("Fraud_rate")
]).sort("Transactions", descending=True).collect()

df_dev_md = df_dev.with_columns([
    pl.col("Fraud_rate").map_elements(lambda x: f"**{x*100:.2f}%**" if x > 0.1 else f"{x*100:.2f}%", return_dtype=pl.String)
])
display(Markdown(df_dev_md.to_pandas().to_markdown(index=False)))

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=df_dev["Device_Category"], y=df_dev["Transactions"], name="Volume", opacity=0.4, marker_color="#00cc96"), secondary_y=False)
fig.add_trace(go.Scatter(x=df_dev["Device_Category"], y=df_dev["Fraud_rate"], name="Fraud Rate", mode="lines+markers", line={"color": "red", "width": 3}), secondary_y=True)
fig.update_layout(title="Transactions and Fraud Rate by DeviceType", hovermode="x unified", template="plotly_white", width=800)
fig.update_yaxes(title_text="Transactions", secondary_y=False)
fig.update_yaxes(title_text="Fraud Rate", tickformat=".1%", secondary_y=True)
fig.show()


| Device_Category   |   Transactions | Fraud_rate   |
|:------------------|---------------:|:-------------|
| _absent_          |         471874 | 2.55%        |
| desktop           |          73450 | 5.56%        |
| mobile            |          45171 | **10.01%**   |
| unknown           |             45 | 2.22%        |